# 03 · Backbone zoo: feature extraction

**Goal.** Freeze three backbones, embed the same tiles, and keep the arrays. Notebooks 04 and 06 only read this cache.

**What you will see.** Embedding size, parameter count, and tiles per second for a random ResNet-50, an ImageNet ResNet-50, and DINOv2 ViT-S/14. DINOv2 also stores the mean of its patch tokens next to the CLS token.

**Runtime.** About 20 minutes in fast mode on a T4 the first time (10k + 2k tiles, three models). A second run hits the `.npy` files and skips the forward pass. Batch size is 256 on CUDA and 32 on CPU. GPU extraction uses `inference_mode` and fp16 autocast.

**Talking point.** Slide 31: a slide is thousands of tiles, so tiles/second is a real decision, not a benchmark trophy.

Run notebook 00 first so the index CSV exists. This notebook reloads that CSV; it does not draw a new subset.


In [1]:
import os
import subprocess
from pathlib import Path
repo = Path("/content/SSL_pathology_demo")
if not (repo / "pyproject.toml").exists():
    subprocess.check_call([
        "git", "clone", "--depth", "1",
        "https://github.com/olonok69/SSL_pathology_demo.git",
        str(repo),
    ])
os.chdir(repo)

In [2]:
MODE = "fast"  # "fast" or "full"

import os
import sys
from pathlib import Path

def _find_repo() -> Path:
    here = Path.cwd().resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "src" / "ssl_demo").is_dir():
            return candidate
    raise FileNotFoundError(
        "Launch this notebook from the ssl-pathology-demo repository "
        "(the folder that contains pyproject.toml)."
    )

REPO = _find_repo()
os.chdir(REPO)
src = str(REPO / "src")
if src not in sys.path:
    sys.path.insert(0, src)

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    import subprocess
    # pyproject.toml does not pin torch, so a Colab CUDA wheel stays installed.
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)])

from ssl_demo.config import SEED, cache_root

CACHE = cache_root()
print(f"MODE={MODE}  SEED={SEED}")
print(f"cache: {CACHE}")


Mounted at /content/drive
MODE=fast  SEED=42
cache: /content/drive/MyDrive/ssl-demo


In [7]:
import subprocess
subprocess.check_call([
    "unzip", "-n",
    "/content/drive/MyDrive/ssl-demo/data/nct/NCT-CRC-HE-100K.zip",
    "NCT-CRC-HE-100K/MUC/*",
    "NCT-CRC-HE-100K/MUS/*",
    "NCT-CRC-HE-100K/NORM/*",
    "NCT-CRC-HE-100K/STR/*",
    "NCT-CRC-HE-100K/TUM/*",
    "-d", "/content/drive/MyDrive/ssl-demo/data/nct",
])

0

In [8]:
from pathlib import Path
root = Path("/content/drive/MyDrive/ssl-demo/data/nct")
for name in ("NCT-CRC-HE-100K", "CRC-VAL-HE-7K"):
    folder = root / name
    print(name, sorted(p.name for p in folder.iterdir() if p.is_dir()))

NCT-CRC-HE-100K ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUC (1)', 'MUS', 'NORM', 'STR', 'TUM']
CRC-VAL-HE-7K ['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUS', 'NORM', 'STR', 'TUM']


In [9]:
from pathlib import Path
folder = Path("/content/drive/MyDrive/ssl-demo/data/nct/NCT-CRC-HE-100K")
print(sorted(p.name for p in folder.iterdir() if p.is_dir()))

['ADI', 'BACK', 'DEB', 'LYM', 'MUC', 'MUC (1)', 'MUS', 'NORM', 'STR', 'TUM']


In [10]:
import torch
import pandas as pd

from ssl_demo.backbones import PHASE1_BACKBONES, backbone_info, get_backbone
from ssl_demo.config import THROUGHPUT_CSV, save_results_csv
from ssl_demo.data import feature_dir, prepare_nct, prepare_pathmnist, prepare_pcam
from ssl_demo.extract import extract_dataset, result_row

#NCLUDE_PCAM = True
NCT_ROOT =  "/content/drive/MyDrive/ssl-demo/data/nct"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 256 if device.type == "cuda" else 32
print("device:", device, "batch:", batch_size)

bundles = {
    "pathmnist": prepare_pathmnist(mode=MODE, cache=CACHE, seed=SEED, size=224, download=True)
}
pcam_index = CACHE / "indices" / f"pcam_train_{MODE}.csv"
if pcam_index.is_file():
    bundles["pcam"] = prepare_pcam(mode=MODE, cache=CACHE, seed=SEED, download=False)
    print("PCam index found, Task B features will be extracted.")
if NCT_ROOT:
    bundles["nct"] = prepare_nct(NCT_ROOT, mode=MODE, cache=CACHE, seed=SEED)
    print("NCT-CRC folder set, full-resolution tiles will be extracted.")

for name in PHASE1_BACKBONES:
    info = backbone_info(name)
    print(f"{name}: dim {info.embed_dim}, {info.kind}, {info.licence}")


device: cuda batch: 256
PCam index found, Task B features will be extracted.
NCT-CRC folder set, full-resolution tiles will be extracted.
random_resnet50: dim 2048, pooled, Randomly initialised torchvision ResNet-50. No pretrained weights.
sup_resnet50_in1k: dim 2048, pooled, torchvision ResNet-50 IMAGENET1K_V1. The checkpoint is downloaded, not stored in this repo. ImageNet content has its own terms.
dinov2_vits14: dim 384, cls+patchmean, DINOv2 (facebookresearch/dinov2), Apache-2.0. Loaded with torch.hub.


Each backbone is loaded with its own normalisation. The supervised ResNet reads the mean and standard deviation off the torchvision checkpoint. DINOv2 uses ImageNet stats and a 224 input, which divides by its patch size of 14. The primary file is the CLS token for the ViT and the pooled vector for the ResNets.


In [11]:
rows = []
for dataset_name, splits in bundles.items():
    for name in PHASE1_BACKBONES:
        info = backbone_info(name)
        model, preprocess, _embed_dim = get_backbone(name)
        for split, source in splits.items():
            result = extract_dataset(
                name=name,
                model=model,
                dataset=source.dataset(preprocess),
                cache_dir=feature_dir(CACHE, dataset_name, split, MODE),
                kind=info.kind,
                device=device,
                batch_size=batch_size,
            )
            rows.append(result_row(result, dataset_name, split))
            if result.cache_hit:
                rate = "cache hit"
            else:
                rate = f"{result.tiles_per_sec:.1f} tiles/s in {result.seconds:.1f}s"
            print(
                f"{dataset_name:10} {split:5} {name:22} "
                f"dim={result.embed_dim:<5} params={result.n_params/1e6:.1f}M  {rate}"
            )
        del model
        if device.type == "cuda":
            torch.cuda.empty_cache()

throughput = pd.DataFrame(rows)
paths = save_results_csv(throughput, THROUGHPUT_CSV, CACHE)
print("wrote", paths[0])
throughput


pathmnist  train random_resnet50        dim=2048  params=23.5M  85.0 tiles/s in 117.6s


pathmnist  test  random_resnet50        dim=2048  params=23.5M  166.4 tiles/s in 12.0s
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 185MB/s]


pathmnist  train sup_resnet50_in1k      dim=2048  params=23.5M  84.2 tiles/s in 118.8s


pathmnist  test  sup_resnet50_in1k      dim=2048  params=23.5M  164.2 tiles/s in 12.2s
Downloading: "https://github.com/facebookresearch/dinov2/zipball/main" to /root/.cache/torch/hub/main.zip


/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/swiglu_ffn.py:51: UserWarning: xFormers is not available (SwiGLU)
  warnings.warn("xFormers is not available (SwiGLU)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/attention.py:33: UserWarning: xFormers is not available (Attention)
  warnings.warn("xFormers is not available (Attention)")
/root/.cache/torch/hub/facebookresearch_dinov2_main/dinov2/layers/block.py:40: UserWarning: xFormers is not available (Block)
  warnings.warn("xFormers is not available (Block)")


... (15 outputs left out)

nct        test  sup_resnet50_in1k      dim=2048  params=23.5M  348.1 tiles/s in 5.7s


nct        train dinov2_vits14          dim=384   params=22.1M  409.2 tiles/s in 24.4s


nct        test  dinov2_vits14          dim=384   params=22.1M  351.3 tiles/s in 5.7s
wrote /content/drive/MyDrive/ssl-demo/results/throughput.csv


,dataset,split,backbone,embed_dim,n_params,n_tiles,seconds,tiles_per_sec,cache_hit
0,pathmnist,train,random_resnet50,2048,23508032,10000,117.602635,85.032108,False
1,pathmnist,test,random_resnet50,2048,23508032,2000,12.016558,166.437013,False
2,pathmnist,train,sup_resnet50_in1k,2048,23508032,10000,118.798591,84.176083,False
3,pathmnist,test,sup_resnet50_in1k,2048,23508032,2000,12.182019,164.176395,False
4,pathmnist,train,dinov2_vits14,384,22056576,10000,115.566092,86.530572,False
5,pathmnist,test,dinov2_vits14,384,22056576,2000,11.648010,171.703156,False
6,pcam,train,random_resnet50,2048,23508032,5000,297.747278,16.792765,False
7,pcam,test,random_resnet50,2048,23508032,2000,17.024782,117.475808,False
8,pcam,train,sup_resnet50_in1k,2048,23508032,5000,10.829697,461.693451,False
9,pcam,test,sup_resnet50_in1k,2048,23508032,2000,4.214892,474.507998,False


## What we learned

- Random, ImageNet, and DINOv2 features now share one cache layout: `features/<dataset>/<mode>/<split>/<backbone>.npy`, plus labels and the source index. Fast and full subsets do not share a folder.
- A second run of this cell loads those files and does not run the network. That is what makes notebooks 04 and 06 short.
- Tiles per second is the number that matters when a whole slide is tens of thousands of tiles. The head has not been fit yet; only the backbone changed.

Table: `results/throughput.csv` (cache copy and repo copy).
